In [4]:
import tensorflow as tf

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Input, GlobalMaxPooling1D
from tensorflow.keras.layers import LSTM, Embedding, TextVectorization
from tensorflow.keras.models import Model

In [7]:
!wget -nc https://lazyprogrammer.me/course_files/spam.csv

--2026-06-20 08:40:39--  https://lazyprogrammer.me/course_files/spam.csv
Resolving lazyprogrammer.me (lazyprogrammer.me)... 172.67.213.166, 104.21.23.210, 2606:4700:3030::ac43:d5a6, ...
Connecting to lazyprogrammer.me (lazyprogrammer.me)|172.67.213.166|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503663 (492K) [text/csv]
Saving to: ‘spam.csv’

spam.csv            100%[===================>] 491.86K  1.99MB/s    in 0.2s    

2026-06-20 08:40:39 (1.99 MB/s) - ‘spam.csv’ saved [503663/503663]



In [9]:
dataset = pd.read_csv("spam.csv", encoding='ISO-8859-1')

In [11]:
dataset.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1, inplace=True)

In [12]:
dataset

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [13]:
dataset.columns = ['labels', 'data']

In [14]:
dataset

,labels,data
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [15]:
dataset['labels'] = dataset['labels'].map({"ham":0, "spam":1})

In [16]:
y = dataset['labels'].values

In [17]:
y

array([0, 0, 1, ..., 0, 0, 0])

In [18]:
x = dataset.drop('labels', axis = 1)

In [19]:
x

,data
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
...,...
5567,This is the 2nd time we have tried 2 contact u...
5568,Will Ì_ b going to esplanade fr home?
5569,"Pity, * was in mood for that. So...any other s..."
5570,The guy did some bitching but I acted like i'd...


In [20]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.33, random_state=42, stratify = y)

In [22]:
train_ds = tf.data.Dataset.from_tensor_slices((xtrain.values, ytrain))
test_ds = tf.data.Dataset.from_tensor_slices((xtest.values, ytest))

In [25]:
MAX_VOCAB_SIZE = 20000
vectorization = TextVectorization(max_tokens=MAX_VOCAB_SIZE)
vectorization.adapt(train_ds.map(lambda x, y: x))

In [29]:
vectorization.get_vocabulary()

['',
 '[UNK]',
 np.str_('to'),
 np.str_('i'),
 np.str_('you'),
 np.str_('a'),
 np.str_('the'),
 np.str_('u'),
 np.str_('and'),
 np.str_('is'),
 np.str_('in'),
 np.str_('my'),
 np.str_('me'),
 np.str_('your'),
 np.str_('for'),
 np.str_('of'),
 np.str_('it'),
 np.str_('have'),
 np.str_('call'),
 np.str_('on'),
 np.str_('that'),
 np.str_('im'),
 np.str_('are'),
 np.str_('now'),
 np.str_('so'),
 np.str_('2'),
 np.str_('but'),
 np.str_('not'),
 np.str_('do'),
 np.str_('can'),
 np.str_('at'),
 np.str_('or'),
 np.str_('ur'),
 np.str_('get'),
 np.str_('be'),
 np.str_('with'),
 np.str_('if'),
 np.str_('just'),
 np.str_('we'),
 np.str_('will'),
 np.str_('this'),
 np.str_('no'),
 np.str_('up'),
 np.str_('when'),
 np.str_('its'),
 np.str_('ltgt'),
 np.str_('go'),
 np.str_('free'),
 np.str_('dont'),
 np.str_('4'),
 np.str_('from'),
 np.str_('ok'),
 np.str_('know'),
 np.str_('what'),
 np.str_('out'),
 np.str_('how'),
 np.str_('all'),
 np.str_('got'),
 np.str_('ill'),
 np.str_('good'),
 np.str_('like

In [27]:
train_ds = train_ds.shuffle(10000).batch(32).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(32).prefetch(tf.data.AUTOTUNE)

In [31]:
V = len(vectorization.get_vocabulary())
D = 64   # embedding dimensionality
M = 256   # number of hidden states
i = Input(shape=(1,), dtype=tf.string)
x = vectorization(i)
x = Embedding(V, D)(x)
x = LSTM(M, return_sequences=True)(x)
x = GlobalMaxPooling1D()(x)
x = Dense(1, activation='sigmoid')(x)
model = Model(i, x)

In [32]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [33]:
model.fit(train_ds, epochs=10, validation_data = test_ds)

Epoch 1/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 27s 206ms/step - accuracy: 0.8768 - loss: 0.3253 - val_accuracy: 0.9636 - val_loss: 0.1562
Epoch 2/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 23s 197ms/step - accuracy: 0.9153 - loss: 0.2108 - val_accuracy: 0.9260 - val_loss: 0.2094
Epoch 3/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 21s 177ms/step - accuracy: 0.9523 - loss: 0.1516 - val_accuracy: 0.9668 - val_loss: 0.1160
Epoch 4/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 23s 194ms/step - accuracy: 0.9775 - loss: 0.0885 - val_accuracy: 0.9690 - val_loss: 0.1050
Epoch 5/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 23s 198ms/step - accuracy: 0.9855 - loss: 0.0628 - val_accuracy: 0.9788 - val_loss: 0.0942
Epoch 6/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 21s 177ms/step - accuracy: 0.9890 - loss: 0.0438 - val_accuracy: 0.9821 - val_loss: 0.0707
Epoch 7/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 22s 191ms/step - accuracy: 0.9968 - loss: 0.0180 - val_accuracy: 0.9831 - val_loss: 0.0843
Epoch 8/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 22s 189ms/step - accuracy: 0.9979 - loss: 0

In [34]:
from sklearn.metrics import f1_score

In [40]:
f1_score(ytest, model.predict(xtest['data'].values) > 0.5)

58/58 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step


0.9423868312757202

In [43]:
xtest.data.values

array(["I'll be late...",
       'Wat makes some people dearer is not just de happiness dat u feel when u meet them but de pain u feel when u miss dem!!!',
       'Remember all those whom i hurt during days of satanic imposter in me.need to pay a price,so be it.may destiny keep me going and as u said pray that i get the mind to get over the same.',
       ...,
       "Sir, I need Velusamy sir's date of birth and company bank facilities details.",
       'Customer Loyalty Offer:The NEW Nokia6650 Mobile from ONLY å£10 at TXTAUCTION! Txt word: START to No: 81151 & get yours Now! 4T&Ctxt TC 150p/MTmsg',
       'Alright took the morphine. Back in yo.'], dtype=object)